# Decision Tree Lab

This notebook explores `DecisionTreeClassifier` and `DecisionTreeRegressor` from scikit-learn across several datasets, covering:

- Baseline fitting and the effect of `max_depth` on accuracy
- Handling missing categorical data (Congressional voting records dataset)
- N-fold cross-validation (Cars evaluation dataset)
- Interpreting induced trees and feature importances
- Techniques to reduce overfitting: max depth, minimum samples per leaf, and cost-complexity pruning (`ccp_alpha`)
- Applying `DecisionTreeRegressor` to a real-valued prediction task (body fat percentage)

Datasets are loaded directly from public URLs so the notebook runs end-to-end without any local setup, aside from the body fat dataset noted below.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn import metrics

In [ ]:
import urllib.request, io

def load_arff_url(url):
    """Load an .arff file directly from a URL (no local download needed)."""
    text = urllib.request.urlopen(url).read().decode('utf-8')
    return arff.loadarff(io.StringIO(text))


## Baseline Fit

### Debug Check
I trained a `DecisionTreeClassifier` with all default parameters on the [Iris dataset](https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/iris.arff) as a sanity check, first decoding the class labels from their raw byte-string encoding.

In [ ]:
data, meta = load_arff_url('https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/iris.arff')

iris = pd.DataFrame(data)
iris['class'] = iris['class'].str.decode('utf-8')

X = iris[['petallength', 'petalwidth', 'sepallength', 'sepalwidth']]
y = iris[['class']]

clf = DecisionTreeClassifier()

clf.fit(X,y)

ypred = clf.predict(X)

score = clf.score(X,y)
print(f'Score: {score}')

### Effect of `max_depth`
I retrained the same classifier with `max_depth=3` to see how constraining tree depth affects accuracy compared to the fully grown tree above.

In [ ]:
clf = DecisionTreeClassifier(max_depth = 3)

clf.fit(X, y)

score = clf.score(X,y)
print(f'Score: {score}')

**Analysis**

Both models had a high accuracy score. With no max depth it got a perfect classification when trained and tested on all the data. When using a max depth of three it wasn't perfect, but had an accuracy of .973, which is pretty high.

## Missing Values, Cross-Validation, and Tree Interpretation

### Handling Missing Values
I used the [Congressional Voting Records dataset](https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/voting_with_missing.arff), which has missing values recorded as `'?'`. Since scikit-learn's CART implementation doesn't accept nominal features directly, I one-hot encoded each feature, effectively treating each missing value as its own category (an "unknown" feature value) rather than dropping or imputing it. I induced the tree fully (no stopping criteria) on an 80/20 train/test split and reported the training and test accuracies.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
voting_data, meta = load_arff_url('https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/voting_with_missing.arff')

voting_df = pd.DataFrame(voting_data)

for column in voting_df:
  voting_df[column] = voting_df[column].str.decode('utf-8')

voting = voting_df.replace('?', 'u')

X = voting.drop('Class', axis=1)
y_v = voting[['Class']]

X_v = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(X_v, y_v, test_size = 0.2, random_state = 42)

clf_v = DecisionTreeClassifier()
clf_v.fit(X_train, y_train)

train_score = clf_v.score(X_train, y_train)
test_score = clf_v.score(X_test, y_test)

print(f'Train score: {train_score}')
print(f'Test score: {test_score}')

**Analysis**

My model got a perfect score on the training data, which makes sense as it's overfitting, and still got a high score on the testing data. It handled missing values by turning them into a new feature value, u, which represents unknown.

### N-Fold Cross-Validation
I applied the decision tree to the [Cars Evaluation dataset](https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/cars.arff) using 10-fold cross-validation, then repeated the process with 5-fold CV to compare the resulting accuracy tables and averages.

In [ ]:
from sklearn.model_selection import cross_val_score

cars, meta = load_arff_url('https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/cars.arff')
cars = pd.DataFrame(cars)

for col in cars:
    cars[col] = cars[col].str.decode('utf-8')
X = cars.drop('class', axis=1)
y = cars['class']
X = pd.get_dummies(X)

clf_c = DecisionTreeClassifier()
clf_c.fit(X, y)

def create_fold_names(array):
  names = []
  for i in range(len(array)):
    names.append(f'Fold {i+1}')
  names.append('Average')
  return names

tenfoldscores = cross_val_score(clf_c, X, y, cv=10)
cell_text_10 = [[score] for score in tenfoldscores]
cell_text_10.append([f"{tenfoldscores.mean()}"])
tenfolds = create_fold_names(tenfoldscores)

fivefoldscores = cross_val_score(clf_c, X, y, cv=5)
cell_text_5 = [[score] for score in fivefoldscores]
cell_text_5.append([f"{fivefoldscores.mean()}"])
fivefolds = create_fold_names(fivefoldscores)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

ax1.axis('tight')
ax1.axis('off')
ax1.set_title('10-Fold Cross Validation')
table10 = ax1.table(cellText=cell_text_10, rowLabels=tenfolds, colLabels=["Accuracy"], loc='center')
table10.scale(1, 1.5)

ax2.axis('tight')
ax2.axis('off')
ax2.set_title('5-Fold Cross Validation')
table5 = ax2.table(cellText=cell_text_5, rowLabels=fivefolds, colLabels=["Accuracy"], loc='center')
table5.scale(1, 1.5)

**Analysis**

N-fold cross validation takes the data and splits into n numbers of folds. It then takes all of those folds except one and uses it to train the data, with the one it not using to train it does the test on. It then does this n number of times so that every fold is used as the test fold. We do it because it gives a more reliable estimate rather than a single train test split. It makes sure every data point is used for both training and testing.
It reveals how sensitive the model is to different sections of the data. We don't end up with a model, we use it for evaluation.

### Interpreting the Trees
For both the Voting and Cars datasets, I visualized the fully induced trees with `tree.plot_tree`, then retrained each with `max_depth=2` to get a simplified view. I compared the top splits the trees learned against each model's `feature_importances_` to see whether they agreed on which features mattered most.

In [ ]:
#Print induced trees for the voting and car data sets
from sklearn import tree


X = voting.drop('Class', axis=1)
y_v = voting[['Class']]

X_v = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(X_v, y_v, test_size = 0.2, random_state = 42)

clf_v = DecisionTreeClassifier()
clf_v.fit(X_train, y_train)





fig, ax = plt.subplots(figsize=(20,10))

tree.plot_tree(clf_v, max_depth=None, feature_names=X_v.columns, filled=True, ax=ax)
ax.set_title('Voting Full Tree', fontsize=25)
fig.tight_layout()

plt.show()

print('\n')



X = cars.drop('class', axis=1)
y = cars['class']
X = pd.get_dummies(X)

clf_c = DecisionTreeClassifier()
clf_c.fit(X, y)






fig, ax = plt.subplots(figsize=(20,10))

tree.plot_tree(clf_c, max_depth=None, feature_names=X.columns, filled=True, ax=ax)
ax.set_title('Cars Full Tree', fontsize=25)
fig.tight_layout()

plt.show()


print('\n')

clf_v = DecisionTreeClassifier(max_depth = 2)
clf_v.fit(X_v, y_v)

fig, ax = plt.subplots(figsize=(20,10))

tree.plot_tree(clf_v, max_depth=None, feature_names=X_v.columns, filled=True, ax=ax)
ax.set_title('Voting Smaller Tree', fontsize=25)
fig.tight_layout()

plt.show()

print('\n')



clf_c = DecisionTreeClassifier(max_depth = 2)
clf_c.fit(X, y)

fig, ax = plt.subplots(figsize=(20,10))

tree.plot_tree(clf_c, max_depth=None, feature_names=X.columns, filled=True, ax=ax)
ax.set_title('Cars Smaller Tree', fontsize=25)
fig.tight_layout()

plt.show()

print('\n')


**Analysis**

Looking at the trees, the voting one figured out that physician-fee-freeze was a big factor, then branched off to adoption-of-the-budget-resolution and synfuels-corporation-cutback. For the cars one it made safety_low as the root, and if that wasn't true, it looked at persons_2. This lines up with feature importances as these are the most influential in the classification.

### Split Criteria
I compared the effect of different split criteria — Gini impurity, entropy, and log-loss — on accuracy, tree structure, and feature importance.

In [ ]:
# Experiment with criterion parameter

cars, meta = load_arff_url('https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/cars.arff')
cars = pd.DataFrame(cars)

for col in cars:
    cars[col] = cars[col].str.decode('utf-8')
X = cars.drop('class', axis=1)
y = cars['class']
X = pd.get_dummies(X)

clf = DecisionTreeClassifier()
clf.fit(X, y)

tenfoldscores = cross_val_score(clf, X, y, cv=10)
print(f'Gini Average Score: {np.mean(tenfoldscores):.4f}')

clf_E = DecisionTreeClassifier(criterion = 'entropy')
clf_E.fit(X, y)

tenfoldscores = cross_val_score(clf_E, X, y, cv=10)
print(f'Entropy Average Score: {np.mean(tenfoldscores):.4f}')

clf_LL = DecisionTreeClassifier(criterion = 'log_loss')
clf_LL.fit(X, y)

tenfoldscores = cross_val_score(clf_LL, X, y, cv=10)
print(f'Log loss Average Score: {np.mean(tenfoldscores):.4f}')


**Analysis**

All of the splits resulted in very, very similar accuracies all performing within 0.2% of each other. This is because all 3 metrics are mathematically designed to favor pure nodes so it results in all of them ranking the same feature as the best splitting point. The tree structure and feature importance remain nearly identical across all 3 critera as they all prioritize the same key attributes.

## Overfitting and Pruning

A fully induced tree on the Cars dataset tends to reach very high training accuracy but can overfit and lose accuracy on the test set. I explored several parameters that constrain tree growth to reduce overfitting.

### Simpler Trees via Growth Constraints
Starting from the fully induced tree (train/test accuracy, node count, and max depth as a baseline), I experimented with parameters that produce smaller, simpler trees:
- `min_samples_leaf`
- `min_samples_split`
- `min_impurity_decrease`

I also tried parameters that constrain the tree more directly, which can risk underfitting if pushed too far:
- `max_depth`
- `max_leaf_nodes`
- `max_features`

For each, I compared train/test accuracy and tree size against the fully induced baseline.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

print('--------------------Fully induced tree-------------------- ')
print(f'Training accuracy: {clf.score(X_train, y_train)}')
print(f'Testing accuracy: {clf.score(X_test, y_test)}')
print(f'Total number of nodes: {clf.tree_.node_count}')
print(f'Maximum tree depth: {clf.tree_.max_depth}')
print('\n')

def different_leafs(val):
    clf = DecisionTreeClassifier(min_samples_leaf = val, random_state=42)
    clf.fit(X_train, y_train)
    print(f'------------------{val} Min samples leaf--------------------')
    print(f'Training accuracy: {clf.score(X_train, y_train)}')
    print(f'Testing accuracy: {clf.score(X_test, y_test)}')
    print(f'Total number of nodes: {clf.tree_.node_count}')
    print(f'Maximum tree depth: {clf.tree_.max_depth}')
    print('\n')

def different_min_splits(val):
    clf = DecisionTreeClassifier(min_samples_split = val, random_state=42)
    clf.fit(X_train, y_train)
    print(f'------------------{val} Min samples split--------------------')
    print(f'Training accuracy: {clf.score(X_train, y_train)}')
    print(f'Testing accuracy: {clf.score(X_test, y_test)}')
    print(f'Total number of nodes: {clf.tree_.node_count}')
    print(f'Maximum tree depth: {clf.tree_.max_depth}')
    print('\n')

def different_impurities(val):
    clf = DecisionTreeClassifier(min_impurity_decrease = val, random_state=42)
    clf.fit(X_train, y_train)
    print(f'------------------{val} Min impurities decrease--------------------')
    print(f'Training accuracy: {clf.score(X_train, y_train)}')
    print(f'Testing accuracy: {clf.score(X_test, y_test)}')
    print(f'Total number of nodes: {clf.tree_.node_count}')
    print(f'Maximum tree depth: {clf.tree_.max_depth}')
    print('\n')

def different_max_depth(val):
    clf = DecisionTreeClassifier(max_depth = val, random_state=42)
    clf.fit(X_train, y_train)
    print(f'------------------{val} Max depth--------------------')
    print(f'Training accuracy: {clf.score(X_train, y_train)}')
    print(f'Testing accuracy: {clf.score(X_test, y_test)}')
    print(f'Total number of nodes: {clf.tree_.node_count}')
    print(f'Maximum tree depth: {clf.tree_.max_depth}')
    print('\n')

def different_max_leaf_nodes(val):
    clf = DecisionTreeClassifier(max_leaf_nodes = val, random_state=42)
    clf.fit(X_train, y_train)
    print(f'------------------{val} Max leaf nodes--------------------')
    print(f'Training accuracy: {clf.score(X_train, y_train)}')
    print(f'Testing accuracy: {clf.score(X_test, y_test)}')
    print(f'Total number of nodes: {clf.tree_.node_count}')
    print(f'Maximum tree depth: {clf.tree_.max_depth}')
    print('\n')

def different_max_features(val):
    clf = DecisionTreeClassifier(max_features = val, random_state=42)
    clf.fit(X_train, y_train)
    print(f'------------------{val} Max features--------------------')
    print(f'Training accuracy: {clf.score(X_train, y_train)}')
    print(f'Testing accuracy: {clf.score(X_test, y_test)}')
    print(f'Total number of nodes: {clf.tree_.node_count}')
    print(f'Maximum tree depth: {clf.tree_.max_depth}')
    print('\n')

for i in range(1, 4):
    different_leafs(i)
    different_min_splits(i + 2)
    different_impurities(i / 1000)
    different_max_depth(i + 2)
    different_max_leaf_nodes(i * 5)
    different_max_features(i)

**Analysis**

The methods used above helped avoid overfitting by restricting the tree's ability to grow which prevented from memorizing training data. I know this because the training accuracy was no longer 1, like it was for the fully induced tree. Training accuracy decreased as the tree structure simplified and testing accuracy improved in several cases. Setting min_samples_leaf to 2 or min_samples_split to 3 helped the most. These got the highest accuracy of (0.962) and shrunk the tree's size.

### Cost-Complexity Pruning
As an alternative to constraining tree growth up front, I induced the Cars tree fully and then pruned it after the fact using [`ccp_alpha`](https://scikit-learn.org/stable/auto_examples/tree/plot_cost_complexity_pruning.html). I tested a range of alpha values and recorded training/test accuracy, node count, and max depth for each, to find the pruning strength that best balanced accuracy and simplicity.

In [ ]:
def different_ccp_alphas(val):
    clf = DecisionTreeClassifier(ccp_alpha = val, random_state=42)
    clf.fit(X_train, y_train)
    print(f'------------------{val} ccp_alpha--------------------')
    print(f'Training accuracy: {clf.score(X_train, y_train)}')
    print(f'Testing accuracy: {clf.score(X_test, y_test)}')
    print(f'Total number of nodes: {clf.tree_.node_count}')
    print(f'Maximum tree depth: {clf.tree_.max_depth}')
    print('\n')

alphas = [0.001, 0.002, 0.005, 0.01, 0.02]

for i in alphas:
    different_ccp_alphas(i)

cell_text_ccp = []
row_labels = []

for a in alphas:
    clf = DecisionTreeClassifier(ccp_alpha=a, random_state=42)
    clf.fit(X_train, y_train)
    train_acc = clf.score(X_train, y_train)
    test_acc = clf.score(X_test, y_test)
    nodes = clf.tree_.node_count
    depth = clf.tree_.max_depth
    cell_text_ccp.append([round(train_acc, 4), round(test_acc, 4), nodes, depth])
    row_labels.append(f"alpha = {a}")

fig, ax = plt.subplots(figsize=(10, 4))

ax.axis('tight')
ax.axis('off')
ax.set_title('ccp_alpha Results')

col_labels = ["Train Accuracy", "Test Accuracy", "Total Nodes", "Max Depth"]

table_ccp = ax.table(cellText=cell_text_ccp, rowLabels=row_labels, colLabels=col_labels, loc='center')

table_ccp.scale(1, 1.5)

plt.show()

**Analysis**

Increasing ccp_alpha simplified the tree structure by reducing the total number of nodes and maximum depth, which lowered training accuracy but improved generalization on the testing.  Instead of limiting the tree before it is created by setting guidelines, the pruning paramater ccp_alpha let the full tree be created first and then cut some parts off. Both approaches successfully reduced overfitting.

## Decision Tree Regression
To apply `DecisionTreeRegressor` to a real-valued target, I picked a body fat percentage dataset (see below) and reported tree statistics (node count, leaf count, max depth), MAE, and R² (coefficient of determination) for both the training and test sets.

In [ ]:
#Learn regression data set
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# Dataset: Body Fat Prediction Dataset (Penrose et al. 1985 / StatLib Archive).
# Download from https://www.kaggle.com/datasets/fedesoriano/body-fat-prediction-dataset
# and place 'bodyfat.csv' in the same directory as this notebook.
body_fat_df = pd.read_csv('bodyfat.csv')

X = body_fat_df.drop(columns=['BodyFat', 'Density'])

#Decided to drop density since bodyfat is calculated from a math equation related to density, the model was cheating

y = body_fat_df['BodyFat']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg = DecisionTreeRegressor(random_state=42)
reg.fit(X_train, y_train)

train_preds = reg.predict(X_train)
test_preds = reg.predict(X_test)

print('--------------------Fully induced tree-------------------- ')
print(f'Total number of nodes: {reg.tree_.node_count}')
print(f'Total number of leaf nodes: {reg.get_n_leaves()}')
print(f'Maximum tree depth: {reg.tree_.max_depth}')

print(f'\nTraining R-squared: {reg.score(X_train, y_train):.4f}')
print(f'Testing R-squared: {reg.score(X_test, y_test):.4f}')

print(f'\nTraining MAE: {mean_absolute_error(y_train, train_preds):.4f}')
print(f'Testing MAE: {mean_absolute_error(y_test, test_preds):.4f}')

#Realized the model was overfitting and doing bad on testing set

alphas = [0.01, 0.1, 0.25, 0.5, 0.75, 2]
cell_text_ccp = []
row_labels = []

for a in alphas:
    reg = DecisionTreeRegressor(ccp_alpha=a, random_state=42)
    reg.fit(X_train, y_train)
    train_preds = reg.predict(X_train)
    test_preds = reg.predict(X_test)
    train_r2 = reg.score(X_train, y_train)
    test_r2 = reg.score(X_test, y_test)
    test_mae = mean_absolute_error(y_test, test_preds)
    nodes = reg.tree_.node_count
    leaves = reg.get_n_leaves()
    depth = reg.tree_.max_depth
    cell_text_ccp.append([round(train_r2, 4), round(test_r2, 4), round(test_mae, 4), nodes, depth])
    row_labels.append(f"alpha = {a}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('tight')
ax.axis('off')
ax.set_title('Body Fat Regression: Cost Complexity Pruning Results')
col_labels = ["Train R-squared", "Test R-squared", "Test MAE", "Total Nodes", "Max Depth"]
table_ccp = ax.table(cellText=cell_text_ccp, rowLabels=row_labels, colLabels=col_labels, loc='center')
table_ccp.scale(1, 1.5)

plt.show()


**Analysis**

I chose to use a bodyfat dataset with features like weight, age, and different measurements of bodyparts (biceps, waist, ankles, etc) because I personally love bodybuilding and thought this would be interesting to see if a ML model could predict bodyfat. Bodyfat is notoriously difficult to estimate, even the most precise measurements available such as DEXA and even MRI can be off by +- 5%.

I first ran the fully induced tree and observed statistics. The model was severly overfitting, as it had a perfect training accuracy and MAE but performed poorly on the testing set (.3389 accuracy and 4.2706 MAE). I decided to prune the tree with different ccp alpha values like I did for the previous problem. I also changed the ccp alpha values because when I ran it with the super low ones almost no change was made, I then realized this was because the MAE was so large that a small a would do almost nothing.

From that, I was able to stop some of the overfitting and had a training score of .8484 and a testing score of .5878 from using a ccp alpha value of 0.5. The test MAE also came down to 3.4926, total nodes of 35 and a max depth of 6. This pruning helped the perfomance. Though not perfect, .5878 is still a pretty good accuracy and was around what I expected as bodyfat is a very hard-to-predict target.